# LangGraph & MCP – Zustandsbehaftete Fabrikagenten

**Internet of Production** · Prof. Dr.-Ing. U. Triltsch · Ostfalia Hochschule für angewandte Wissenschaften

---

## Lernziele

Nach Abschluss dieses Notebooks sind Sie in der Lage:

1. Die Kernkonzepte von **LangGraph** (State Graph, Knoten, Kanten, Checkpointing) zu erläutern.
2. Das bekannte MESA-Fabrikszenario (`MachineAgent ↔ CoordinatorAgent ↔ MaintenanceAgent`) auf einen LangGraph-Workflow abzubilden.
3. Die Rolle von **MCP (Model Context Protocol)** als Konnektivitätsschicht zwischen LLM-basierten Agenten und externen Werkzeugen bzw. Datenquellen zu verstehen.
4. Regelbasiertes reaktives Verhalten (MESA) von LLM-augmentiertem zustandsbehaftetem Verhalten (LangGraph) zu unterscheiden.

---

## Einordnung: Wo steht dieses Notebook?

| Notebook | Agentenparadigma | Kommunikation | LLM? |
|---|---|---|---|
| `01_Reactive_Agents_solution.ipynb` | Reaktiv (Sense–Decide–Act) | Lokaler Grid-Zustand | Nein |
| `04_Reactive_Agents_OPC_solution.ipynb` | Kommunikativ | OPC UA (Shared Bus) | Nein |
| **Dieses Notebook** | **Zustandsbehaftet / LLM-augmentiert** | **Gemeinsamer Graph-Zustand + MCP** | **Ja (optional)** |

Wir verwenden dasselbe **Fabrikszenario**: Maschinen erhitzen sich, der Koordinator überwacht HOT-Zustände, der Wartungsagent repariert Maschinen. Was sich ändert, ist das *Architekturmuster* — vom MESA-Simulationsloop zu einer **LangGraph-Zustandsmaschine**.

## 1. Konzeptuelle Grundlagen

### 1.1 LangGraph — Kernideen

LangGraph modelliert einen Multi-Agenten-Workflow als **gerichteten Graphen**:

```
Knoten A ──(Kantenbedingung)──► Knoten B ──► Knoten C
           ▲                                     │
           └────────────(Rückschleife)───────────┘
```

- **State (Zustand)**: ein typisiertes Dictionary, das von allen Knoten geteilt wird (analog zum OPC UA-Adressraum aus den vorigen Notebooks).
- **Nodes (Knoten)**: Python-Funktionen (oder LLM-gestützte Agenten), die den Zustand *lesen*, Arbeit verrichten und Aktualisierungen *zurückschreiben*.
- **Edges (Kanten)**: Routing-Logik — welcher Knoten läuft als nächstes? Kann bedingt sein (entspricht reaktiven Regeln).
- **Checkpointing**: Der Zustand ist persistierbar; der Graph kann unterbrochen und fortgesetzt werden (Human-in-the-Loop).

### 1.2 MCP — Model Context Protocol

MCP ist die standardisierte **Konnektivitätsschicht** zwischen Agenten und externen Werkzeugen:

```
 LLM-Agent (Host)
      │
   MCP-Client ──JSON-RPC 2.0──► MCP-Server: OPC UA  ──► SPS / SCADA
   MCP-Client ──────────────► MCP-Server: REST-API  ──► ERP / MES
   MCP-Client ──────────────► MCP-Server: Datenbank ──► TimescaleDB
```

Kernaussage: Der Agent benötigt **keinen eigenen Integrationscode** pro Werkzeug — er ruft eine standardisierte MCP-Schnittstelle auf. Die Serverseite kapselt die protokollspezifische Logik (OPC UA, REST, SQL).

### 1.3 Konzeptuelle Abbildung: MESA → LangGraph

| MESA-Konzept | LangGraph-Äquivalent |
|---|---|
| `agent.step()` | Graph-Knotenfunktion |
| `model.grid` / `model.schedule` | Gemeinsames `GraphState`-Dict |
| `DataCollector` | Zustandsfelder (z. B. `hot_machines`) |
| OPC UA-Subscription-Callback | Bedingte Kantenroutierung |
| `MaintenanceAgent.choose_target()` | CoordinatorAgent-Knotenlogik |
| Mesa `RandomActivation`-Scheduler | LangGraph-Ausführungsreihenfolge |

## 2. Installation

LangGraph ist Teil des LangChain-Ökosystems. Wir installieren nur die minimalen Abhängigkeiten für dieses Beispiel (kein externer LLM-API-Schlüssel für die regelbasierte Variante erforderlich).

In [ ]:
# LangGraph und LangChain Core installieren
!pip install -q langgraph>=0.2 langchain-core>=0.2

In [ ]:
import random
import time
from typing import TypedDict, List, Dict, Literal

from langgraph.graph import StateGraph, END

print("LangGraph-Import erfolgreich.")

## 3. Definition des gemeinsamen Graph-Zustands

Der `GraphState` ist die **einzige Quelle der Wahrheit** für alle Agenten im Graphen —
analog zum OPC UA-Adressraum im Notebook zu kommunikativen Agenten.

Jeder Knoten empfängt den vollständigen Zustand und gibt eine *partielle Aktualisierung* zurück (nur die Felder, die er verändert).

In [ ]:
class MachineStatus(TypedDict):
    """Maschinenstatus-Eintrag im gemeinsamen Zustand."""
    name: str
    temperature: float
    state: str          # "OK" | "WARM" | "HOT"
    repair_needed: bool


class GraphState(TypedDict):
    """Gemeinsamer Zustand für den Fabrikagenten-Graphen.
    
    Alle Agenten lesen aus diesem Dictionary und schreiben in dieses Dictionary.
    LangGraph führt partielle Aktualisierungen automatisch zusammen.
    """
    # Maschinentelemetrie (aktualisiert durch den MachineAgent-Knoten)
    machines: Dict[str, MachineStatus]
    
    # Koordinatorentscheidungen (aktualisiert durch den CoordinatorAgent-Knoten)
    repair_queue: List[str]        # Liste der Maschinennamen, die auf Reparatur warten
    coordinator_log: List[str]     # Prüfprotokoll der Koordinatorentscheidungen
    
    # Wartungsfortschritt (aktualisiert durch den MaintenanceAgent-Knoten)
    maintenance_log: List[str]
    
    # Steuerungsfelder
    cycle: int                     # Zähler der Simulationsschritte
    max_cycles: int
    next_action: str               # wird vom Router-Edge ausgewertet


# ── Hilfsfunktion: Initialzustand für N Maschinen ────────────────────────────
def initial_state(n_machines: int = 5, max_cycles: int = 10) -> GraphState:
    machines = {}
    for i in range(n_machines):
        name = f"M{i+1:02d}"
        machines[name] = MachineStatus(
            name=name,
            temperature=20.0,
            state="OK",
            repair_needed=False,
        )
    return GraphState(
        machines=machines,
        repair_queue=[],
        coordinator_log=[],
        maintenance_log=[],
        cycle=0,
        max_cycles=max_cycles,
        next_action="machine_node",
    )

print("GraphState definiert.")
print("Beispiel-Initialzustand für 3 Maschinen:")
s = initial_state(3, max_cycles=5)
for k, v in s.items():
    print(f"  {k}: {v}")

## 4. Agenten-Knoten

Jeder Agent wird als **einfache Python-Funktion** implementiert, die:
1. den aktuellen `GraphState` empfängt,
2. ihre Logik ausführt und
3. eine *partielle Zustandsaktualisierung* zurückgibt (nur die veränderten Felder).

Dieses Muster ist bei allen LangGraph-Knoten identisch — unabhängig davon, ob sie regelbasiert (wie hier) oder LLM-gestützt sind.

### 4.1 MachineAgent-Knoten

Simuliert die Temperaturmessung für alle Maschinen (entspricht `MachineAgent.sense_temperature()` + `MachineAgent.decide()` in den MESA-Notebooks).

In [ ]:
SCHWELLENWERT = 70.0  # °C


def machine_node(state: GraphState) -> dict:
    """MachineAgent-Knoten: Temperatur messen und Zustand aller Maschinen aktualisieren.
    
    Entspricht der MESA-Methode MachineAgent.step(), jedoch auf alle
    Maschinen in einer einzigen Knotenausführung angewendet.
    """
    aktualisierte_maschinen = {}
    
    for name, m in state["machines"].items():
        temp = m["temperature"]
        
        # Messen: zufälligen thermischen Drift addieren (simulierte Sensormessung)
        if m["state"] != "HOT":
            rauschen = random.uniform(-1.0, 5.0)
            temp = temp + rauschen
        
        # Entscheiden: Zustand anhand des Schwellenwerts klassifizieren
        if temp > SCHWELLENWERT:
            neuer_zustand = "HOT"
        elif temp > 0.75 * SCHWELLENWERT:
            neuer_zustand = "WARM"
        elif temp < 20.0:
            neuer_zustand = "COOL"
        else:
            neuer_zustand = "OK"
        
        aktualisierte_maschinen[name] = MachineStatus(
            name=name,
            temperature=round(temp, 1),
            state=neuer_zustand,
            repair_needed=m["repair_needed"],  # hier unverändert
        )
        print(f"  [MachineAgent] {name}: {round(temp,1)} °C → {neuer_zustand}")
    
    return {"machines": aktualisierte_maschinen, "next_action": "coordinator_node"}


print("machine_node definiert.")

### 4.2 CoordinatorAgent-Knoten

Liest alle Maschinenzustände aus dem gemeinsamen Zustand, erkennt HOT-Maschinen und befüllt die `repair_queue`. Dies ist die *Orchestrierungslogik* — in einem Produktivsystem könnte dieser Knoten ein LLM aufrufen, um die Priorisierung zu begründen (siehe Abschnitt 6).

Beachten Sie die Analogie zur Funktion `sync_repair_jobs_from_opcua()` im Notebook zu kommunikativen Agenten: Der Koordinator fungiert dort ebenso als Brücke zwischen Sensorik und Aktion.

In [ ]:
def coordinator_node(state: GraphState) -> dict:
    """CoordinatorAgent-Knoten: HOT-Maschinen erkennen und Reparaturwarteschlange verwalten.
    
    Entscheidungsregeln:
    - HOT-Maschinen in die repair_queue aufnehmen, sofern noch nicht vorhanden.
    - Maschinen aus der Warteschlange entfernen, wenn sie nicht mehr HOT sind.
    - Für jede Entscheidung einen Protokolleintrag schreiben.
    """
    zyklus = state["cycle"]
    bestehende_queue = list(state["repair_queue"])
    protokoll = list(state["coordinator_log"])
    maschinen = state["machines"]
    
    neue_queue = []
    
    # Aus der Warteschlange entfernen, wenn nicht mehr HOT (bereits repariert)
    for name in bestehende_queue:
        if maschinen[name]["state"] == "HOT":
            neue_queue.append(name)
        else:
            protokoll.append(f"[Zyklus {zyklus}] [KOORD] {name} aus der Reparaturwarteschlange entfernt.")
    
    # Neu HOT gewordene Maschinen hinzufügen
    for name, m in maschinen.items():
        if m["state"] == "HOT" and name not in neue_queue:
            neue_queue.append(name)
            protokoll.append(
                f"[Zyklus {zyklus}] [KOORD] {name} bei {m['temperature']} °C → REPARATUR EINGEPLANT."
            )
            print(f"  [CoordinatorAgent] {name} HOT → in Reparaturwarteschlange aufgenommen.")
    
    # repair_needed-Flags in den Maschinenzuständen aktualisieren
    aktualisierte_maschinen = {}
    for name, m in maschinen.items():
        aktualisierte_maschinen[name] = MachineStatus(
            name=m["name"],
            temperature=m["temperature"],
            state=m["state"],
            repair_needed=(name in neue_queue),
        )
    
    naechste_aktion = "maintenance_node" if neue_queue else "advance_cycle"
    
    return {
        "machines": aktualisierte_maschinen,
        "repair_queue": neue_queue,
        "coordinator_log": protokoll,
        "next_action": naechste_aktion,
    }


print("coordinator_node definiert.")

### 4.3 MaintenanceAgent-Knoten

Verarbeitet die Reparaturwarteschlange: setzt Temperatur und Zustand der ersten Maschine in der Queue zurück.
In den MESA-Notebooks bewegte sich der `MaintenanceAgent` physisch durch das Gitter;
hier werden "Bewegung" und "Reparatur" zu einer einzigen Zustandsaktualisierung zusammengefasst
(der Graph abstrahiert die räumliche Dimension).

In [ ]:
def maintenance_node(state: GraphState) -> dict:
    """MaintenanceAgent-Knoten: erste Maschine in der Reparaturwarteschlange reparieren.
    
    Vereinfachung für Lehrzwecke: eine Reparatur pro Graphzyklus.
    In einem Produktivsystem könnten mehrere Wartungsagenten parallel laufen
    (LangGraph Multi-Actor-Muster).
    """
    queue = list(state["repair_queue"])
    maschinen = dict(state["machines"])
    protokoll = list(state["maintenance_log"])
    zyklus = state["cycle"]
    
    if not queue:
        return {"next_action": "advance_cycle"}
    
    # Erste Maschine in der Warteschlange reparieren
    ziel = queue.pop(0)
    
    # Maschine zurücksetzen (entspricht: agent.temperature = 20.0; agent.state = "OK")
    maschinen[ziel] = MachineStatus(
        name=ziel,
        temperature=20.0,
        state="OK",
        repair_needed=False,
    )
    
    eintrag = f"[Zyklus {zyklus}] [WARTUNG] {ziel} repariert → Temperatur auf 20,0 °C zurückgesetzt."
    protokoll.append(eintrag)
    print(f"  [MaintenanceAgent] {eintrag}")
    
    return {
        "machines": maschinen,
        "repair_queue": queue,
        "maintenance_log": protokoll,
        "next_action": "advance_cycle",
    }


def advance_cycle_node(state: GraphState) -> dict:
    """Zyklusnummer erhöhen. Fungiert als Schleifensteuerknoten."""
    neuer_zyklus = state["cycle"] + 1
    print(f"\n─── Ende von Zyklus {state['cycle']} ───\n")
    return {"cycle": neuer_zyklus, "next_action": "machine_node"}


print("maintenance_node und advance_cycle_node definiert.")

## 5. Aufbau des LangGraph-Zustandsgraphen

Wir fügen nun die Knoten zusammen und verbinden sie mit **bedingten Kanten**.
Die Routing-Funktion liest `state["next_action"]` — von jedem Knoten gesetzt —
um zu bestimmen, welcher Knoten als nächster ausgeführt wird.

```
START
  │
  ▼
machine_node  ──────────────────────────────────────────────►
  │                                                    coordinator_node
  │                                                          │
  │                          HOT-Maschinen?  ─── JA ──► maintenance_node
  │                                                          │
  │               ◄───────── NEIN ─── advance_cycle_node ◄───┘
  │                                              │
  └────────── Zyklus < max_cycles ◄──────────────┘
                     │
              Zyklus >= max_cycles
                     │
                    END
```

In [ ]:
# ── Router: liest next_action aus dem Zustand ────────────────────────────────
def router(state: GraphState) -> Literal[
    "machine_node", "coordinator_node", "maintenance_node", "advance_cycle", "__end__"
]:
    """Routet zum nächsten Knoten basierend auf state['next_action'] und Zyklusanzahl."""
    if state["cycle"] >= state["max_cycles"]:
        return "__end__"
    return state["next_action"]


# ── Graphen aufbauen ─────────────────────────────────────────────────────────
builder = StateGraph(GraphState)

# Knoten hinzufügen
builder.add_node("machine_node",     machine_node)
builder.add_node("coordinator_node", coordinator_node)
builder.add_node("maintenance_node", maintenance_node)
builder.add_node("advance_cycle",    advance_cycle_node)

# Einstiegspunkt
builder.set_entry_point("machine_node")

# Bedingte Kanten (alle Knoten verwenden denselben Router)
for knoten in ["machine_node", "coordinator_node", "maintenance_node", "advance_cycle"]:
    builder.add_conditional_edges(
        knoten,
        router,
        {
            "machine_node":     "machine_node",
            "coordinator_node": "coordinator_node",
            "maintenance_node": "maintenance_node",
            "advance_cycle":    "advance_cycle",
            "__end__":          END,
        },
    )

# Graphen kompilieren
graph = builder.compile()

print("Graph erfolgreich kompiliert.")
print("Knoten:", list(graph.nodes))

## 5.1 Simulation ausführen

In [ ]:
# Initialzustand: 5 Maschinen, 8 Simulationszyklen
zustand = initial_state(n_machines=5, max_cycles=30)

print("=" * 60)
print("FABRIKAGENTEN-SIMULATION — LangGraph")
print(f"Maschinen: {len(zustand['machines'])}  |  Max. Zyklen: {zustand['max_cycles']}")
print("=" * 60)

# Graphen bis END ausführen
endzustand = graph.invoke(zustand)

print("\n" + "=" * 60)
print("SIMULATION ABGESCHLOSSEN")
print("=" * 60)

In [ ]:
# ── Abschließende Maschinenzustände anzeigen ──────────────────────────────────
print("\nAbschließende Maschinenzustände:")
print(f"{'Maschine':<10} {'Temperatur':>13} {'Zustand':<8} {'Reparatur nötig'}")
print("-" * 50)
for name, m in endzustand["machines"].items():
    print(f"{name:<10} {m['temperature']:>10.1f} °C {m['state']:<8} {m['repair_needed']}")

# ── Koordinatorprotokoll ──────────────────────────────────────────────────────
print("\nKoordinatorprotokoll:")
for eintrag in endzustand["coordinator_log"]:
    print(" ", eintrag)

# ── Wartungsprotokoll ─────────────────────────────────────────────────────────
print("\nWartungsprotokoll:")
for eintrag in endzustand["maintenance_log"]:
    print(" ", eintrag)

## 6. Erweiterung: LLM-gestützter CoordinatorAgent (konzeptuell)

Der obige Koordinatorknoten verwendet **regelbasierte Logik**. In einem Produktivsystem erfordert die Ersetzung durch einen LLM-Aufruf lediglich eine Änderung des Knotenrumpfs:

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

def coordinator_node_llm(state: GraphState) -> dict:
    # Relevanten Zustand für das LLM serialisieren
    maschinenzusammenfassung = [
        f"{name}: {m['temperature']} °C, Zustand={m['state']}"
        for name, m in state['machines'].items()
    ]
    prompt = (
        "Sie sind ein Fabrikkoordinatoragent. "
        "Entscheiden Sie anhand der folgenden Maschinentelemetrie, welche Maschinen "
        "repariert werden müssen, und begründen Sie Ihre Priorisierung.\n\n"
        + "\n".join(maschinenzusammenfassung)
    )
    antwort = llm.invoke([HumanMessage(content=prompt)])
    # LLM-Antwort auswerten und Zustand entsprechend aktualisieren ...
    ...
```

**Kernbeobachtung**: Der Rest des Graphen (`machine_node`, `maintenance_node`, Kanten) bleibt unverändert. Dies ist der architekturelle Vorteil von LangGraph — Knoten sind unabhängig voneinander austauschbar.

### 6.1 MCP-Integration (konzeptuell)

Wenn der Koordinator statt aus dem simulierten Zustand von einem echten OPC UA-Server lesen soll, stellt ein MCP-Server, der den OPC UA-Endpunkt kapselt, die standardisierte Werkzeugschnittstelle bereit:

```python
# Konzeptuell: MCP-Werkzeugaufruf innerhalb eines LangGraph-Knotens
# Der Agent ruft das MCP-Werkzeug auf;
# der Server übernimmt die OPC UA-Protokolldetails.

werkzeuge = [
    {
        "type": "mcp_tool",
        "server": "opcua-fabrik-server",
        "name": "read_machine_temperature",
        "input": {"machine_id": "M01"}
    }
]
# Der MCP-Server übersetzt dies in:
#   asyncua.Client.get_node(node_id).read_value()
# ... und gibt das Ergebnis an den Agenten zurück.
```

Dies spiegelt die in der Vorlesung gezeigte Architektur wider:
**LLM-Agent (Host) → MCP-Client → MCP-Server: OPC UA → SPS/SCADA**

## 7. Vergleichsanalyse: MESA vs. LangGraph

| Dimension | MESA (Reaktiv/Kommunikativ) | LangGraph |
|---|---|---|
| **Zustandsverwaltung** | Pro-Agenten-Attribute + modellweiter DataCollector | Zentrales typisiertes `GraphState`-Dict, persistierbar |
| **Ausführungssteuerung** | Scheduler (RandomActivation usw.) | Explizite Graphtopologie + bedingte Kanten |
| **Kommunikation** | Gemeinsames Gitter / OPC UA-Adressraum | Gemeinsames Zustands-Dict (+ optionales MCP für externe Daten) |
| **Beobachtbarkeit** | DataCollector → pandas DataFrame | Zustandssnapshot an jedem Knoten; vollständiges Checkpointing |
| **LLM-Integration** | Nicht nativ unterstützt | Erstklassig (jeder Knoten kann ein LLM aufrufen) |
| **Human-in-the-Loop** | Nicht unterstützt | Eingebaut (Unterbrechung vor/nach jedem Knoten) |
| **Skalierbarkeit** | Tausende Agenten auf einem Rechner | Begrenzt durch LLM-Latenz; geeignet für ~10–100 Agenten |
| **Lernkurve** | Gering (reines Python, MESA-API) | Mittel (Graphabstraktionen, LangChain-Ökosystem) |
| **Beste Einsatzgebiete** | Populationsebene, räumliche Dynamik | Orchestrierung, komplexe Entscheidungslogik, LLM-Reasoning |

**Fazit**: MESA und LangGraph sind komplementär, nicht konkurrierend. MESA ist das richtige Werkzeug für die Simulation großer Populationen einfacher Agenten. LangGraph eignet sich, wenn einzelne Agenten persistenten Zustand, komplexes Entscheidungs-Reasoning, LLM-Augmentierung oder menschliche Aufsicht benötigen — genau die Anforderungen einer produktionsreifen IIoT-Orchestrierungsschicht.

## 8. Aufgaben

### 8.1 Routing-Logik anpassen
- Erweitern Sie den `coordinator_node` so, dass WARM-Maschinen in eine separate *Vorwarnliste* aufgenommen werden (ohne sofortige Reparatur auszulösen).
- Fügen Sie einen neuen Knoten `alert_node` hinzu, der für WARM-Maschinen eine Warnung ausgibt.
- Aktualisieren Sie den Router entsprechend.

### 8.2 Parallele Wartungsagenten
- Derzeit wird pro Zyklus eine Maschine repariert. Modifizieren Sie `maintenance_node` so, dass bis zu zwei Maschinen pro Zyklus repariert werden.
- Wie beeinflusst dies die Anzahl der HOT-Maschinen im Zeitverlauf?

### 8.3 Checkpointing (fortgeschritten)
- Installieren Sie `langgraph[sqlite]` und fügen Sie dem kompilierten Graphen einen `MemorySaver`-Checkpointer hinzu.
- Unterbrechen Sie die Simulation nach 3 Zyklen, inspizieren Sie den Checkpoint, ändern Sie eine Maschinentemperatur manuell und setzen Sie die Simulation fort.
- Dies demonstriert das **Human-in-the-Loop**-Muster aus den Vorlesungsfolien.

```python
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
```

### 8.4 MCP-Server-Stub (fortgeschritten)
- Implementieren Sie einen minimalen Python-`FastAPI`-Server, der einen `GET /temperature/{machine_id}`-Endpunkt bereitstellt.
- Modifizieren Sie den `machine_node` so, dass er diesen HTTP-Endpunkt anstatt der lokalen Rauschsimulation aufruft.
- Diskutieren Sie: Wie würden Sie diesen REST-Aufruf durch einen OPC UA-Lesezugriff über einen MCP-Server ersetzen?

## 9. Zusammenfassung

Dieses Notebook hat gezeigt:

1. **LangGraph-Zustandsgraph** als produktionsreife Alternative zum MESA-Simulationsloop für die Agentenorchestrierung.
2. **Typisierter gemeinsamer Zustand** (`GraphState`) als architekturelles Äquivalent des OPC UA-Adressraums — eine einzige Quelle der Wahrheit für alle Agenten.
3. **Bedingte Kantenroutierung** als Äquivalent reaktiver Regeln in MESA-Agenten auf Graphebene.
4. **MCP** als standardisierte Integrationsschicht zur Verbindung von LLM-Agenten mit OPC UA-Servern, REST-APIs und Datenbanken — ohne werkzeugspezifischen Integrationscode.
5. Das **Austauchprinzip**: Jeder regelbasierte Knoten kann ohne Änderung der umgebenden Graphstruktur durch einen LLM-gestützten Knoten ersetzt werden.

---

**Literatur und Ressourcen**

- LangGraph-Dokumentation: https://langchain-ai.github.io/langgraph/
- Model Context Protocol: https://modelcontextprotocol.io
- Google A2A-Framework: https://codelabs.developers.google.com/intro-a2a-purchasing-concierge
- n8n Workflow-Automatisierung: https://n8n.io
- MESA Multi-Agenten-Framework: https://mesa.readthedocs.io/en/stable/